# Notebook 09: Multimodal - Image-to-Text (Image Captioning)

**Learning Objectives:**
- Generate text descriptions from images
- Use BLIP (Bootstrapping Language-Image Pre-training)
- Understand multimodal models
- Apply to image accessibility and content understanding

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **small (CPU-friendly)** | Salesforce/blip-image-captioning-base | 990MB | 6GB | 6GB RAM, CPU | Good quality |
| **large (GPU-optimized)** | Salesforce/blip-image-captioning-large | 1.9GB | 8GB | 10GB VRAM (RTX 4080) | Better captions |

### Software Requirements
- Python 3.8+
- Libraries: `transformers`, `torch`, `PIL`

In [ ]:
import sys
import random
import time

import numpy as np
import torch
import transformers
from transformers import AutoProcessor, BlipForConditionalGeneration, pipeline
from PIL import Image
import requests
from io import BytesIO
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Expected Behaviors

### First Time Running
- **Model Download**: ~990MB for BLIP-base (~5-8 minutes)
- Large model combining vision and language
- Cached for subsequent runs

### Setup Cell Output
```
PyTorch version: 2.x.x
CUDA available: True/False
```

### Model Loading
```
Model loaded on: cpu (or cuda)
```
- **CPU**: 15-20 seconds (large multimodal model)
- **GPU**: 8-12 seconds

### Caption Output
- Returns natural language description
- Example: `"a cat sitting on a couch looking at the camera"`

### Caption Quality
- **Clear, single-subject images**: Very accurate, descriptive
- **Complex scenes**: Captures main elements, may miss details
- **Multiple objects**: Describes most prominent objects
- **Actions**: Often captures what's happening in scene

### Expected Caption Length
- **Unconditional**: 5-15 words typically
- **With prompt**: Longer, more specific descriptions
- Controlled by `max_length` parameter

### Performance
- **Single image**:
  - CPU: 5-8 seconds
  - GPU: 1-2 seconds
- **Batch of 5 images**:
  - CPU: 20-30 seconds
  - GPU: 4-6 seconds

### Caption Style
- **Factual and descriptive**
- Uses common language
- Focuses on visible elements
- Sometimes includes colors, positions, activities

### Conditional Captioning
- Can provide text prompts to guide captions
- Example prompts: "a photograph of", "this image shows"
- Helps steer caption style and content

### Sampling for Variety
- `do_sample=True` generates diverse captions
- Same image can produce different valid captions
- Useful for creative applications

### Common Observations
- Accurate for common objects/scenes (people, animals, vehicles)
- May hallucinate details not actually present
- Sometimes generic for unusual images
- Better on photos than drawings/artwork

### Multimodal Understanding
- Combines vision (what's in image) + language (how to describe it)
- Trained on millions of image-caption pairs
- Can describe relationships ("person holding phone")

## Overview

**Image captioning** is a multimodal task that generates natural language descriptions of images. It sits at the intersection of computer vision and natural language processing, requiring the model to both understand visual content and express it in coherent text.

### How BLIP Works

BLIP (Bootstrapping Language-Image Pre-training) uses a **vision-language architecture** with three key components:

1. **Vision Encoder**: A Vision Transformer (ViT) that converts the input image into a sequence of visual embeddings
2. **Text Decoder**: A language model that generates captions token-by-token, conditioned on the visual embeddings
3. **Image-Text Matching**: A contrastive module that aligns visual and textual representations during pre-training

### Unconditional vs Conditional Captioning

- **Unconditional**: The model generates a caption from scratch, describing what it "sees" in the image
- **Conditional**: You provide a text prompt (e.g., "a photograph of") and the model completes the description, allowing you to guide the caption style

### Key Concepts

- **Multimodal model**: A model that processes multiple data types (images + text) in a unified architecture
- **Visual embeddings**: Dense vector representations of image regions, analogous to word embeddings in NLP
- **Cross-attention**: The mechanism that allows the text decoder to "look at" relevant parts of the image while generating each word

## Setup and Installation

The imports cell above loads all necessary libraries. BLIP requires a **processor** (handles image preprocessing and tokenization) and a **model** (the vision-language transformer). We also import `requests` and `PIL` for loading images from URLs.

## Model Selection

In [ ]:
# CHOOSE YOUR MODEL:

# Option 1: small model (CPU-friendly)
MODEL_NAME = "Salesforce/blip-image-captioning-base"  # 990MB

# Option 2: large model (GPU-optimized, better quality)
# MODEL_NAME = "Salesforce/blip-image-captioning-large"  # 1.9GB

print(f"Selected model: {MODEL_NAME}")

## Method 1: Pipeline API

The Pipeline API provides the simplest way to caption images. It handles model loading, image preprocessing, and caption generation in one call.

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL.

    Args:
        url: URL of the image to download.

    Returns:
        PIL Image in RGB format.
    """
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    return img


# Create image-to-text pipeline
print("Creating captioning pipeline...")
try:
    captioner = pipeline(
        "image-to-text",
        model=MODEL_NAME,
        device=device,
    )
    print("Pipeline ready")
except Exception as e:
    print(f"Error creating pipeline: {e}")
    captioner = None

In [ ]:
# Caption a single image using the pipeline
IMAGE_URL = "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=500"
image = load_image_from_url(IMAGE_URL)

result = captioner(image)
print(f"Image size: {image.size}")
print(f"\n=== PIPELINE CAPTION ===")
print(result[0]["generated_text"])

image

## Method 2: Manual Model Loading

Loading the model and processor manually gives you finer control over generation parameters like `max_length`, `num_beams`, `do_sample`, and `temperature`. This is essential for conditional captioning and producing diverse captions.

In [ ]:
# Load model and processor manually
print(f"Loading {MODEL_NAME} manually...")
try:
    blip_processor = AutoProcessor.from_pretrained(MODEL_NAME)
    blip_model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
    print(f"Model loaded on: {device}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Ensure you have internet access for first-time model download.")

In [ ]:
def generate_caption(
    image: Image.Image,
    max_length: int = 50,
) -> str:
    """Generate an unconditional caption for an image.

    Args:
        image: PIL Image to caption.
        max_length: Maximum number of tokens in the caption.

    Returns:
        Generated caption string.
    """
    inputs = blip_processor(images=image, return_tensors="pt").to(device)
    generated_ids = blip_model.generate(**inputs, max_length=max_length)
    caption = blip_processor.decode(generated_ids[0], skip_special_tokens=True)
    return caption


def generate_conditional_caption(
    image: Image.Image,
    prompt_text: str,
    max_length: int = 50,
) -> str:
    """Generate a caption conditioned on a text prompt.

    Args:
        image: PIL Image to caption.
        prompt_text: Text prompt to guide caption generation.
        max_length: Maximum number of tokens in the caption.

    Returns:
        Generated caption string.
    """
    inputs = blip_processor(
        images=image, text=prompt_text, return_tensors="pt"
    ).to(device)
    generated_ids = blip_model.generate(**inputs, max_length=max_length)
    caption = blip_processor.decode(generated_ids[0], skip_special_tokens=True)
    return caption


# Unconditional caption
caption = generate_caption(image)
print(f"=== UNCONDITIONAL CAPTION ===\n{caption}")

# Conditional captions with different prompts
prompts = ["a photograph of", "this image shows", "the picture depicts"]

print("\n=== CONDITIONAL CAPTIONS ===")
for prompt in prompts:
    caption = generate_conditional_caption(image, prompt)
    print(f"\nPrompt: '{prompt}'")
    print(f"Caption: {caption}")

In [ ]:
def generate_multiple_captions(
    image: Image.Image,
    num_captions: int = 3,
    max_length: int = 50,
    temperature: float = 0.7,
) -> list[str]:
    """Generate multiple diverse captions for an image using sampling.

    Args:
        image: PIL Image to caption.
        num_captions: Number of caption variations to generate.
        max_length: Maximum number of tokens per caption.
        temperature: Sampling temperature (higher = more diverse).

    Returns:
        List of generated caption strings.
    """
    inputs = blip_processor(images=image, return_tensors="pt").to(device)

    captions: list[str] = []
    for _ in range(num_captions):
        generated_ids = blip_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            do_sample=True,
            temperature=temperature,
        )
        caption = blip_processor.decode(generated_ids[0], skip_special_tokens=True)
        captions.append(caption)

    return captions


# Test with a landscape image
landscape_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=500"
landscape_image = load_image_from_url(landscape_url)
captions = generate_multiple_captions(landscape_image, num_captions=3)

print("=== MULTIPLE CAPTION VARIATIONS ===")
for i, caption in enumerate(captions, 1):
    print(f"{i}. {caption}")

## Practical Applications

Image captioning has real-world uses in **accessibility** (generating alt-text for visually impaired users), **content moderation** (describing uploaded images automatically), and **media cataloging** (tagging and searching large image libraries by their content). Below we test the model on local images from `sample_data/` and then benchmark performance across different image types.

In [ ]:
import os

sample_data_path = os.path.join("..", "sample_data")

if os.path.exists(sample_data_path):
    image_files = [
        f for f in os.listdir(sample_data_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    if image_files:
        print("=== CAPTIONING LOCAL IMAGES ===")
        for img_file in image_files[:3]:
            img_path = os.path.join(sample_data_path, img_file)
            img = Image.open(img_path).convert("RGB")
            caption = generate_caption(img)
            print(f"\n{img_file}: {caption}")
    else:
        print("No images found in sample_data/. Add some to test!")
else:
    print("sample_data/ folder not found — skipping local image demo.")

## Performance Benchmarking

Let's measure captioning speed across different image types and generation settings to understand what affects latency. This helps you plan for production use cases where response time matters.

In [ ]:
def benchmark_captioning(
    image_urls: list[str],
    labels: list[str],
) -> pd.DataFrame:
    """Benchmark captioning speed across multiple images.

    Args:
        image_urls: List of image URLs to benchmark.
        labels: Descriptive label for each image.

    Returns:
        DataFrame with timing results and generated captions.
    """
    results: list[dict[str, str | float]] = []

    for url, label in zip(image_urls, labels):
        img = load_image_from_url(url)

        start = time.time()
        caption = generate_caption(img)
        elapsed = time.time() - start

        results.append({
            "Image": label,
            "Size": f"{img.size[0]}x{img.size[1]}",
            "Time (s)": round(elapsed, 2),
            "Caption": caption,
        })

    return pd.DataFrame(results)


# Benchmark on diverse image types
benchmark_urls = [
    "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=500",
    "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=500",
    "https://images.unsplash.com/photo-1555685812-4b943f1cb0eb?w=500",
]
benchmark_labels = ["Cat portrait", "Mountain landscape", "City street"]

results_df = benchmark_captioning(benchmark_urls, benchmark_labels)
print(f"Device: {device}")
print(f"Model: {MODEL_NAME}\n")
print(results_df.to_string(index=False))

## State-of-the-Art Vision-Language Models

BLIP is a solid foundation, but newer models go well beyond simple captioning — supporting visual question answering, instruction following, and multi-turn image chat. These require significantly more hardware (24-80GB VRAM), so they are out of scope for this notebook.

| Model | Size | VQA Score | Key Capability | Best For |
|-------|------|-----------|----------------|----------|
| **BLIP** (this notebook) | 990MB | 77.5% | Captioning, retrieval | Learning basics |
| **BLIP-2** | 5.5GB | 82.2% | VQA, captioning | Efficient VL tasks |
| **InstructBLIP** | 13GB | 82.8% | Instruction following | Flexible task handling |
| **LLaVA** | 13GB | 80.0% | Visual chat, reasoning | Conversational AI |
| **Qwen-VL** | 20GB | 78.5% | Multilingual, OCR | Chinese + English |
| **CogVLM** | 20GB | 83.6% | SOTA performance | Research, benchmarks |

**Learning path**: BLIP (here) -> BLIP-2 (VQA) -> InstructBLIP (instructions) -> LLaVA (chat) -> CogVLM (research)

## Exercises

1. **Diverse Images**: Test with various image types (animals, landscapes, objects, people)
2. **Quality Assessment**: Compare base vs large model captions
3. **Custom Images**: Caption your own photos
4. **Caption Length**: Experiment with `max_length` parameter
5. **Batch Processing**: Process multiple images efficiently

In [ ]:
# Exercise 1: Caption diverse image types
exercise_urls = [
    "https://images.unsplash.com/photo-1474511320723-9a56873571b7?w=500",  # Animal
    "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=500",  # Cityscape
]

print("=== EXERCISE: DIVERSE IMAGE CAPTIONS ===")
for url in exercise_urls:
    img = load_image_from_url(url)
    unconditional = generate_caption(img)
    conditional = generate_conditional_caption(img, "a detailed photo of")
    print(f"\nUnconditional: {unconditional}")
    print(f"Conditional:   {conditional}")

# Exercise 2: Experiment with max_length
print("\n=== EXERCISE: CAPTION LENGTH ===")
test_image = load_image_from_url(exercise_urls[0])
for length in [20, 50, 100]:
    caption = generate_caption(test_image, max_length=length)
    print(f"max_length={length:3d}: {caption}")

## Key Takeaways

✅ **BLIP** bridges vision and language for image captioning

✅ **Multimodal models** process both images and text

✅ Can generate **unconditional** or **conditional** captions

✅ Useful for **accessibility** and **content understanding**

✅ Sampling generates diverse captions for same image

## Next Steps

- Try **Notebook 10**: Ollama Integration
- Explore [vision-language models](https://huggingface.co/models?pipeline_tag=image-to-text)
- Learn about Visual Question Answering (VQA)

## Resources

- [BLIP Paper](https://arxiv.org/abs/2201.12086)
- [Image-to-Text Guide](https://huggingface.co/docs/transformers/tasks/image_captioning)
- [BLIP Model Card](https://huggingface.co/Salesforce/blip-image-captioning-base)